# SVTRv2X Run 2 (Tier 1 Architecture Injection): Bilingual Latin + Indic Training Pipeline

This notebook trains the **SVTRv2X Tier 1** scene text recognition model featuring:
1. **Squeeze-and-Excitation (SE) Stem**: Dynamic channel weighting to boost fine diacritic strokes (matras, headlines, dots) and suppress image background noise.
2. **Dual 3x3 ConvResidualBlocks**: Enhances receptive field depth before downsampling, preserving sub-glyph spatial features.
3. **Learnable 2D Positional Embeddings**: Injects explicit 2D $(H, W)$ coordinate layouts for vertically stacked Brahmic conjuncts.
4. **MoE Top-1 Routing + Self-Distillation (SDM)**: Preserves Latin word recognition while scaling to 12 Indic languages.
5. **Warm-Start Migration**: Automatically adapts and transfers learned weights from Run 1's `best.pth` checkpoint directly into Tier 1!

### Step 1: Verify Hardware Accelerator (GPU)

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name: {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

### Step 2: Mount Google Drive (Persistent Storage)

In [ ]:
import os, shutil
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# Checkpoint directory specifically for Run 2 Tier 1
tier1_checkpoint_dir = '/content/drive/MyDrive/svtrv2x_checkpoints/mixed_latin_indic_tiny_run2_tier1'
os.makedirs(tier1_checkpoint_dir, exist_ok=True)
print(f'✓ Persistent checkpoint directory ready: {tier1_checkpoint_dir}')

### Step 3: Clone / Synchronize AMSTR Repository

In [ ]:
%cd /content
import os

if not os.path.exists('/content/AMSTR'):
    !git clone https://github.com/GodishalaAshwith/AMSTR.git
%cd /content/AMSTR
!git fetch origin main
!git reset --hard origin/main
!git log -1 --oneline

### Step 4: Install Dependencies

In [ ]:
!pip install -q lmdb pyclipper pyyaml rapidfuzz tqdm huggingface_hub torchvision datasets pyarrow

### Step 5: Dataset Download & Cache Verification (Idempotent)

In [ ]:
import os
from huggingface_hub import snapshot_download

# 1. Download Union14M-L (Latin) if not already downloaded
union_check = '/content/AMSTR/Union14M-L-LMDB-Filtered/filter_train_challenging'
if not os.path.exists(union_check):
    print('Downloading Union14M-L-LMDB-Filtered...')
    snapshot_download(
        repo_id='topdu/OpenOCR-Data',
        repo_type='dataset',
        allow_patterns=['Union14M-L-LMDB-Filtered/**', 'evaluation/**'],
        local_dir='/content/AMSTR'
    )
else:
    print('✓ Union14M-L dataset already cached.')

# 2. Download IndicSTR12 crops if LMDB not already built
indic_lmdb_path = '/content/AMSTR/IndicSTR12/indicstr12_lmdb'
indic_raw_path = '/content/AMSTR/IndicSTR12/indicstr12_train'
if not os.path.exists(indic_lmdb_path) and not os.path.exists(indic_raw_path):
    print('Downloading IndicSTR12 crops...')
    snapshot_download(
        repo_id='darknight054/indicstr12-crops',
        repo_type='dataset',
        local_dir=indic_raw_path
    )
else:
    print('✓ IndicSTR12 data already present.')

### Step 6: Build Indic LMDB & Merged Character Dictionary (Idempotent)

In [ ]:
import os, io, glob, lmdb
import pyarrow.parquet as pq
from PIL import Image
from tqdm import tqdm

lmdb_out = '/content/AMSTR/IndicSTR12/indicstr12_lmdb'
merged_dict_path = 'tools/utils/mixed_latin_indic_dict.txt'

if not os.path.exists(lmdb_out) or not os.path.exists(merged_dict_path):
    os.makedirs(lmdb_out, exist_ok=True)
    env = lmdb.open(lmdb_out, map_size=1099511627776)

    parquet_files = sorted(glob.glob('/content/AMSTR/IndicSTR12/indicstr12_train/**/*.parquet', recursive=True))
    print(f'Packing {len(parquet_files)} Indic Parquet files into LMDB...')

    indic_chars = set()
    idx = 0

    with env.begin(write=True) as txn:
        for p_file in tqdm(parquet_files, desc='Converting to LMDB'):
            table = pq.read_table(p_file)
            pydict = table.to_pydict()
            images = pydict.get('image', [])
            texts = pydict.get('text', pydict.get('label', []))

            for img_obj, label in zip(images, texts):
                if not label or len(str(label).strip()) == 0:
                    continue

                if isinstance(img_obj, dict) and 'bytes' in img_obj and img_obj['bytes'] is not None:
                    img_bytes = img_obj['bytes']
                elif isinstance(img_obj, bytes):
                    img_bytes = img_obj
                else:
                    continue

                try:
                    img = Image.open(io.BytesIO(img_bytes))
                    w, h = img.size
                    if w * h == 0 or w < 8 or h < 8:
                        continue
                except Exception:
                    continue

                text_str = str(label).strip()
                indic_chars.update(list(text_str))

                idx += 1
                txn.put(f'image-{idx:09d}'.encode(), img_bytes)
                txn.put(f'label-{idx:09d}'.encode(), text_str.encode('utf-8'))
                txn.put(f'wh-{idx:09d}'.encode(), f'{w}_{h}'.encode('utf-8'))

        txn.put('num-samples'.encode(), str(idx).encode())

    env.close()
    print(f'✓ Created Indic LMDB with {idx:,} samples at {lmdb_out}')

    # Build merged Latin + Indic character dictionary
    with open('tools/utils/EN_symbol_dict.txt', 'r', encoding='utf-8') as f:
        latin_chars = [line.strip('\n').strip('\r') for line in f if line.strip()]

    all_chars = sorted(list(set(latin_chars).union(indic_chars)))
    with open(merged_dict_path, 'w', encoding='utf-8') as f:
        for ch in all_chars:
            f.write(f'{ch}\n')
    print(f'✓ Created merged character dictionary with {len(all_chars)} symbols at {merged_dict_path}')
else:
    print(f'✓ Indic LMDB and merged dictionary already present at {lmdb_out}')

### Step 7: Unit Test & Validate Tier 1 Architecture Injection

In [ ]:
!python tools/test_svtrv2x_tier1.py

### Step 8: Launch Training (with Automatic Warm-Start from Run 1 `best.pth`)

In [ ]:
import os, glob, shutil, torch

tier1_dir = '/content/drive/MyDrive/svtrv2x_checkpoints/mixed_latin_indic_tiny_run2_tier1'
run1_dir = '/content/drive/MyDrive/svtrv2x_checkpoints/mixed_latin_indic_tiny'
os.makedirs(tier1_dir, exist_ok=True)

config_path = 'configs/rec/svtrv2/svtrv2x_tiny_mixed_latin_indic_run2_tier1.yml'
cmd = f'python tools/train_rec.py -c {config_path}'

# 1. First priority: Resume from existing Tier 1 checkpoint if already training
tier1_latest = os.path.join(tier1_dir, 'latest.pth')
tier1_best = os.path.join(tier1_dir, 'best.pth')

if os.path.exists(tier1_latest):
    print(f'>>> Resuming active Tier 1 training from: {tier1_latest}')
    cmd += f' -o Global.checkpoints={tier1_latest}'
elif os.path.exists(tier1_best):
    print(f'>>> Resuming active Tier 1 training from: {tier1_best}')
    cmd += f' -o Global.checkpoints={tier1_best}'
# 2. Second priority: Warm-start fine-tuning from Run 1's best model (70k steps, 83.20% Indic acc)
elif os.path.exists(os.path.join(run1_dir, 'best.pth')):
    run1_best = os.path.join(run1_dir, 'best.pth')
    print(f'>>> Found Run 1 best checkpoint at {run1_best}!')
    print('>>> Warm-starting Tier 1 via architectural weight adaptation (LR=0.00015 fine-tuning)...')
    cmd += f' -o Global.pretrained_model={run1_best} Optimizer.lr=0.00015'
else:
    print('>>> No previous checkpoint found. Training Tier 1 from scratch with AdamW + OneCycleLR.')

print(f'Executing:\n{cmd}\n')
!{cmd}

### Step 8.5: Dedicated Resume Block (Continue Interrupted Training / Disconnect Recovery)
Run this cell whenever your session disconnects, times out, or when you want to resume Tier 1 training. It automatically auto-repairs any BatchNorm statistics and resumes from the exact latest or best checkpoint.

In [ ]:
import os, glob, shutil, torch

# ----------------------------------------------------------------------
# RESUME SELECTION:
# 'latest': Resumes from the most recent training step (RECOMMENDED for continuing training)
# 'best'  : Resumes from the highest recorded validation accuracy checkpoint
# ----------------------------------------------------------------------
RESUME_CHOICE = 'latest'

tier1_dir = '/content/drive/MyDrive/svtrv2x_checkpoints/mixed_latin_indic_tiny_run2_tier1'
config_path = 'configs/rec/svtrv2/svtrv2x_tiny_mixed_latin_indic_run2_tier1.yml'

print('=' * 75)
print(f'Inspecting Tier 1 Checkpoints in: {tier1_dir}')
print('=' * 75)

if not os.path.exists(tier1_dir):
    raise FileNotFoundError(f'Directory {tier1_dir} not found! Check your Google Drive mount.')

ckpts = [f for f in os.listdir(tier1_dir) if f.endswith('.pth')]
valid_ckpts = {}

for ckpt_name in sorted(ckpts):
    ckpt_path = os.path.join(tier1_dir, ckpt_name)
    try:
        data = torch.load(ckpt_path, map_location='cpu')
        state_dict = data.get('state_dict', {})
        if len(state_dict) == 0:
            continue
            
        # Auto-repair BatchNorm tracking buffers if any abnormal values exist
        repaired = False
        for k, v in state_dict.items():
            if torch.isnan(v).any() or torch.isinf(v).any():
                if 'running_mean' in k:
                    v.zero_()
                    repaired = True
                elif 'running_var' in k:
                    v.fill_(1.0)
                    repaired = True
                elif 'num_batches_tracked' in k:
                    v.zero_()
                    repaired = True
                else:
                    v[torch.isnan(v) | torch.isinf(v)] = 0.0
                    repaired = True
        if repaired:
            torch.save(data, ckpt_path)
            print(f'  ✓ Auto-repaired BatchNorm running statistics in {ckpt_name}')
            
        step = data.get('global_step', 0)
        epoch = data.get('epoch', 0)
        metrics = data.get('metrics', {})
        acc = metrics.get('acc', metrics.get('gtc_acc', 0.0))
        valid_ckpts[ckpt_name] = {'path': ckpt_path, 'step': step, 'epoch': epoch, 'acc': acc}
        print(f'  ✓ {ckpt_name:<16} | Epoch: {epoch:<3} | Step: {step:<7} | Val Acc: {acc*100:.2f}%')
    except Exception as e:
        print(f'  ⚠ Could not parse {ckpt_name}: {e}')

# Select target checkpoint
target_ckpt = None
if RESUME_CHOICE == 'latest' and 'latest.pth' in valid_ckpts:
    target_ckpt = valid_ckpts['latest.pth']
elif RESUME_CHOICE == 'best' and 'best.pth' in valid_ckpts:
    target_ckpt = valid_ckpts['best.pth']
elif len(valid_ckpts) > 0:
    target_ckpt = max(valid_ckpts.values(), key=lambda x: x['step'])

if target_ckpt is not None:
    print('-' * 75)
    print(f'>>> Resuming Tier 1 Training from: {target_ckpt["path"]}')
    print(f'>>> Starting from Epoch {target_ckpt["epoch"]}, Step {target_ckpt["step"]}')
    print('-' * 75)
    
    cmd = f'python tools/train_rec.py -c {config_path} -o Global.checkpoints={target_ckpt["path"]}'
    print(f'Executing command:\n{cmd}\n')
    !{cmd}
else:
    print('✗ No valid checkpoints found to resume in Tier 1 directory.')

### Step 9: Benchmark & Qualitative Evaluation (IndicSTR12 + Latin Benchmarks)

In [ ]:
import os, torch
from tools.engine.trainer import Trainer
from tools.utils.config import ArgsParser

# Select best Tier 1 checkpoint
tier1_eval_ckpt = '/content/drive/MyDrive/svtrv2x_checkpoints/mixed_latin_indic_tiny_run2_tier1/best.pth'
if not os.path.exists(tier1_eval_ckpt):
    tier1_eval_ckpt = '/content/drive/MyDrive/svtrv2x_checkpoints/mixed_latin_indic_tiny_run2_tier1/latest.pth'

print(f'Evaluating Tier 1 Checkpoint: {tier1_eval_ckpt}')

# Helper for clean text extraction
def extract_text(item):
    while isinstance(item, (list, tuple)):
        item = item[0]
    return str(item).strip()

# Initialize evaluation trainer on IndicSTR12
FLAGS = ArgsParser().parse_args([
    '-c', 'configs/rec/svtrv2/svtrv2x_tiny_mixed_latin_indic_run2_tier1.yml',
    '-o',
    f'Global.checkpoints={tier1_eval_ckpt}',
    'Eval.dataset.data_dir_list=["IndicSTR12/indicstr12_lmdb"]'
])

trainer = Trainer(FLAGS.opt)
eval_res = trainer.eval()

print('\n' + '=' * 70)
print('SVTRv2X TIER 1 EVALUATION RESULTS ON INDICSTR12')
print('=' * 70)
for k, v in eval_res.items():
    if isinstance(v, float):
        print(f'  • {k:<25}: {v*100:.2f}%')
print('=' * 70)

# Qualitative sample prints
trainer.model.eval()
valid_loader = trainer.valid_dataloader

print(f'\n{"Ground Truth (Indic)":<28} | {"CTC Prediction":<28} | {"GTC Transformer":<28} | {"Status"}')
print('-' * 105)

samples_printed = 0
with torch.no_grad():
    for batch in valid_loader:
        images = batch[0].cuda()
        batch_numpy = [t.numpy() for t in batch]
        preds = trainer.model(images, data=None)
        post_res = trainer.post_process_class(preds, batch_numpy)
        
        if isinstance(post_res, (list, tuple)) and len(post_res) == 2 and isinstance(post_res[0], (list, tuple)):
            gtc_preds, _ = post_res[0]
            ctc_preds, targets = post_res[1]
        else:
            ctc_preds, targets = post_res
            gtc_preds = ctc_preds
            
        for c_item, g_item, t_item in zip(ctc_preds, gtc_preds, targets):
            c_text = extract_text(c_item)
            g_text = extract_text(g_item)
            tgt_text = extract_text(t_item)
            
            c_ok = (c_text == tgt_text)
            g_ok = (g_text == tgt_text)
            status = '✓ Exact Match' if (c_ok and g_ok) else ('★ CTC Match' if c_ok else ('★ GTC Match' if g_ok else '✗ Mismatch'))
            print(f'{tgt_text:<28} | {c_text:<28} | {g_text:<28} | {status}')
            
            samples_printed += 1
            if samples_printed >= 25:
                break
        if samples_printed >= 25:
            break
print('-' * 105)